# Object Weighing
This notebook contains scripts, code samples, notes, and systems designed to work as the part of the algorithm that assigns weights to signals/objects detected in the environment.

## Project updates
After doing some more research into the future of this project, I have realized that there is a chance that all the prompt parsing code I had written might not be needed at all. I will be now trying to use `sentence_transformers` to convert the prompt and all input signals (assuming it can run at a decent speed) into vectors and comparing them directly. If this works, it would be amazing but it might require a lot of tuning and playing around with it.

## Using Sentence Transformers
The python library, `sentence_transformers` (will be referred to as `st`), is used to create text embeddings which can then be compared for similarity.

Below, is the example provided on the `st` documentation page, rewritten a little bit to be better example for this project. On top of the base example, I also added in `cosine_similarity` checks using `sklearn`.

`cosine_similarity` basically calculates the cosine of the angle between two vectors, which in this case are the embeddings of the input prompt and the signals, to calculate how similar the two embeddings are. This is better than what I was doing earlier with `sapCy` because this can very easily compare meanings without having to parse anything.

I put in a bunch of prompt types, including the full prompt and also some semi parsed variations of it. I also put in various signals including those that are very related and those that are very unrelated.

The first model that was tried was `all-MiniLM-L6-v2`. This was working pretty slow on my hardware and it also didn't *seem* accurate. Therefore, I moved to the `paraphrase-albert-small-v2` model which was significantly faster and also *seemed* more accurate.

In [17]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("paraphrase-albert-small-v2")

prompt = [
    "10 year old boy in red winter clothing lost in the dense forest",
    "red winter clothing",
    "10 year old boy",
    "lost"
    ]

signals = [
    "red t-shirt",
    "green t-shirt",
    "red jacket",
    "red rock",
    "tree",
    "footprints"
]


promptEmbeddings = model.encode(prompt)
signalEmbeddings = model.encode(signals)

similarities = model.similarity(promptEmbeddings, signalEmbeddings)
similarity_score = cosine_similarity(promptEmbeddings, signalEmbeddings)

print(similarity_score)

[[0.29502857 0.04555189 0.39761537 0.2656671  0.36944643 0.20626926]
 [0.46212783 0.14639768 0.8075857  0.36514312 0.12200657 0.1730755 ]
 [0.25633937 0.23143648 0.20242494 0.22088914 0.11701734 0.10262364]
 [0.18960993 0.09980679 0.26922095 0.2846457  0.16981971 0.20770115]]
